https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/

여기서는 날씨를 확인할 수 있는 간단한 ReAct 에이전트 앱을 만들 것이다. 이 앱은 에이전트(LLM)와 도구들로 구성된다. 앱과 대화할 때, 먼저 에이전트(LLM)를 호출하여 도구를 사용할지 여부를 결정한다. 그런 다음 다음과 같은 루프를 실행한다.

에이전트가 행동을 취하라고 말한 경우(즉, 도구를 호출해야 할 경우), 도구를 실행하고 결과를 에이전트에 전달한다.
에이전트가 도구를 실행하라고 하지 않은 경우, 작업을 완료하고 사용자에게 응답한다.

https://rudaks.tistory.com/entry/langgraph-ReAct-에이전트를-사용하는-방법 [[루닥스 블로그] 연습만이 살길이다:티스토리]

In [ ]:
!pip install langgraph langchain-openai
!pip install python-dotenv
!pip install -U typing_extensions langchain langchain-core

In [ ]:
import os
from dotenv import load_dotenv

# 바로 임포트
# os.environ["OPENAI_API_KEY"] ="sk-proj-IeiPd3N4ZP..."

# .env 파일 로드
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print(api_key)

In [ ]:
# First we initialize the model we want to use.
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# For this tutorial we will use custom tool that returns pre-defined values for weather in two cities (NYC & SF)

from typing import Literal

from langchain_core.tools import tool


@tool
def get_weather(city: Literal["서울", "부산"]):
    """Use this to get weather information."""
    if city == "서울":
        return "서울은 흐릴것 같아요"
    elif city == "부산":
        return "부산은 항상 맑아요"
    else:
        raise AssertionError("Unknown city")


tools = [get_weather]


# Define the graph

from langgraph.prebuilt import create_react_agent

graph = create_react_agent(model, tools=tools)

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png(
            output_file_path="how-to-create-react-agent.png"
        )
    )
)

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [ ]:
inputs = {"messages": [("user", "what is the weather in Busan?")]}
print_stream(graph.stream(inputs, stream_mode="values"))

In [ ]:
inputs = {"messages": [("user", "who built you?")]}
print_stream(graph.stream(inputs, stream_mode="values"))